In [53]:
import pandas as pd

df = pd.read_csv("Customer-Churn-Records.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RowNumber           10000 non-null  int64  
 1   CustomerId          10000 non-null  int64  
 2   Surname             10000 non-null  object 
 3   CreditScore         10000 non-null  int64  
 4   Geography           10000 non-null  object 
 5   Gender              10000 non-null  object 
 6   Age                 10000 non-null  int64  
 7   Tenure              10000 non-null  int64  
 8   Balance             10000 non-null  float64
 9   NumOfProducts       10000 non-null  int64  
 10  HasCrCard           10000 non-null  int64  
 11  IsActiveMember      10000 non-null  int64  
 12  EstimatedSalary     10000 non-null  float64
 13  Exited              10000 non-null  int64  
 14  Complain            10000 non-null  int64  
 15  Satisfaction Score  10000 non-null  int64  
 16  Card 

In [54]:
df = df.drop(["RowNumber","CustomerId", "Surname", "Complain","Satisfaction Score" ], axis=1)

In [55]:
df.isna().sum()
df.duplicated().sum()

np.int64(0)

In [56]:
categ_col = df.select_dtypes(include="object").columns
df = pd.get_dummies(df, columns=categ_col, dtype="int")

In [57]:
#creating the model
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report
X = df.drop("Exited", axis=1)
y = df["Exited"]
X_train,  X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify = y,
    random_state=42
)

model = DecisionTreeClassifier(
    max_depth=6,
    criterion="entropy",
    min_samples_split=50,
    min_samples_leaf=20,
    class_weight={0:1, 1:2.96},
    random_state=42
)
scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("cross val score: ",np.array(scores).mean())
print("accuracy: ",accuracy_score(y_test, y_pred))
print("confusion matrix: ", confusion_matrix(y_test, y_pred))
print("classification report: ", classification_report(y_test, y_pred))

cross val score:  0.7893
accuracy:  0.7715
confusion matrix:  [[1221  371]
 [  86  322]]
classification report:                precision    recall  f1-score   support

           0       0.93      0.77      0.84      1592
           1       0.46      0.79      0.58       408

    accuracy                           0.77      2000
   macro avg       0.70      0.78      0.71      2000
weighted avg       0.84      0.77      0.79      2000



In [58]:
from sklearn.metrics import roc_auc_score
probs = model.predict_proba(X_test)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, probs))

ROC-AUC: 0.8539796100601044
